In [12]:
%%writefile search.cu

#include <iostream>
#include <vector>
#include <random>
#include <chrono>
#include <cuda_runtime.h>
#include <iomanip>
#include <algorithm>

using namespace std;

struct PatternRef {
    int pat_idx;
    int byte_pos;
};


void runCpuSearch(const vector<uint8_t>& haystack,
                  const vector<vector<uint8_t>>& needles,
                  vector<uint8_t>& hits) {
    const int H = (int)haystack.size();
    const int N = (int)needles.size();
    hits.assign(N, 0);

    for (int p = 0; p < N; ++p) {
        const auto& pat = needles[p];
        const int L = (int)pat.size();
        for (int i = 0; i <= H - L; ++i) {
            int k = 0;
            while (k < L && haystack[i + k] == pat[k]) ++k;
            if (k == L) { hits[p] = 1; break; }
        }
    }
}


__global__ void kernelPerPattern(const uint8_t* __restrict__ hay, int hlen,
                                  const uint8_t* __restrict__ pats, const int* __restrict__ plens,
                                  int npats, uint8_t* __restrict__ out) {
    int pid = blockIdx.x * blockDim.x + threadIdx.x;
    if (pid >= npats) return;

    int L = plens[pid];
    const uint8_t* p = pats + pid * 256;

    for (int s = 0; s <= hlen - L; ++s) {
        int k = 0;
        while (k < L && hay[s + k] == p[k]) ++k;
        if (k == L) { out[pid] = 1; return; }
    }
}


__global__ void kernelVoteMatrix(const uint8_t* __restrict__ hay, int hlen,
                                  const PatternRef* __restrict__ idx, const int* __restrict__ bounds,
                                  int* vote, int npats) {
    int pos = blockIdx.x * blockDim.x + threadIdx.x;
    if (pos >= hlen) return;

    uint8_t c = hay[pos];
    for (int e = bounds[c]; e < bounds[c + 1]; ++e) {
        PatternRef ref = idx[e];
        int anchor = pos - ref.byte_pos;
        if (anchor >= 0 && anchor < hlen)
            atomicSub((unsigned int*)&vote[ref.pat_idx * hlen + anchor], 1);
    }
}


void buildTestCase(vector<uint8_t>& hay, int hsize,
                   vector<vector<uint8_t>>& needles, int npats,
                   int lo, int hi) {
    mt19937 rng(42);
    uniform_int_distribution<int> anyByte(0, 255);
    uniform_int_distribution<int> anyLen(lo, hi);
    uniform_int_distribution<int> anyPos(0, hsize - lo - 1);

    hay.resize(hsize);
    for (auto& b : hay) b = (uint8_t)anyByte(rng);

    needles.resize(npats);
    for (int i = 0; i < npats; ++i) {
        int len = anyLen(rng);
        needles[i].resize(len);
        if (i % 5 != 0) {
            int st = anyPos(rng);
            for (int j = 0; j < len; ++j) needles[i][j] = hay[st + j];
        } else {
            for (auto& b : needles[i]) b = (uint8_t)anyByte(rng);
        }
    }
}


int main() {
    cout << "=========================================\n"
         << "  Multi-Pattern Search  |  CUDA Lab\n"
         << "=========================================\n\n";

    {
        int dc; cudaGetDeviceCount(&dc);
        if (dc > 0) {
            cudaDeviceProp pr; cudaGetDeviceProperties(&pr, 0);
            cout << "Device : " << pr.name << "\n"
                 << "CC     : " << pr.major << "." << pr.minor << "\n"
                 << "VRAM   : " << pr.totalGlobalMem / (1 << 20) << " MB\n\n";
        }
    }

    constexpr int HSIZE  = 100000;
    constexpr int NPATS  = 2000;
    constexpr int LO_LEN = 3;
    constexpr int HI_LEN = 15;
    constexpr int SLOT   = 256;

    cout << "Haystack : " << HSIZE  << " B\n"
         << "Patterns : " << NPATS  << "\n"
         << "Lengths  : " << LO_LEN << " – " << HI_LEN << " B\n\n";

    vector<uint8_t> hay;
    vector<vector<uint8_t>> needles;
    buildTestCase(hay, HSIZE, needles, NPATS, LO_LEN, HI_LEN);

    vector<uint8_t> cpuHits;
    auto t0 = chrono::high_resolution_clock::now();
    runCpuSearch(hay, needles, cpuHits);
    double cpuMs = chrono::duration<double, milli>(
                       chrono::high_resolution_clock::now() - t0).count();
    int cpuFound = (int)count(cpuHits.begin(), cpuHits.end(), (uint8_t)1);
    cout << "[CPU]   " << fixed << setprecision(2) << cpuMs << " ms"
         << "  found " << cpuFound << "/" << NPATS << "\n\n";

    vector<uint8_t> flatPats(NPATS * SLOT, 0);
    vector<int>     patLens(NPATS);
    for (int i = 0; i < NPATS; ++i) {
        patLens[i] = (int)needles[i].size();
        for (int j = 0; j < patLens[i]; ++j)
            flatPats[i * SLOT + j] = needles[i][j];
    }

    uint8_t *dHay, *dPats, *dOutA;
    int     *dLens;
    cudaMalloc(&dHay,  HSIZE * sizeof(uint8_t));
    cudaMalloc(&dPats, NPATS * SLOT * sizeof(uint8_t));
    cudaMalloc(&dLens, NPATS * sizeof(int));
    cudaMalloc(&dOutA, NPATS * sizeof(uint8_t));
    cudaMemcpy(dHay,  hay.data(),       HSIZE * sizeof(uint8_t),       cudaMemcpyHostToDevice);
    cudaMemcpy(dPats, flatPats.data(),  NPATS * SLOT * sizeof(uint8_t),cudaMemcpyHostToDevice);
    cudaMemcpy(dLens, patLens.data(),   NPATS * sizeof(int),            cudaMemcpyHostToDevice);
    cudaMemset(dOutA, 0, NPATS * sizeof(uint8_t));

    const int TPB = 256;
    cudaEvent_t evA0, evA1, evB0, evB1;
    cudaEventCreate(&evA0); cudaEventCreate(&evA1);
    cudaEventCreate(&evB0); cudaEventCreate(&evB1);

    cudaEventRecord(evA0);
    kernelPerPattern<<<(NPATS + TPB - 1) / TPB, TPB>>>(
        dHay, HSIZE, dPats, dLens, NPATS, dOutA);
    cudaEventRecord(evA1);
    cudaEventSynchronize(evA1);

    float msA; cudaEventElapsedTime(&msA, evA0, evA1);
    vector<uint8_t> hitsA(NPATS);
    cudaMemcpy(hitsA.data(), dOutA, NPATS * sizeof(uint8_t), cudaMemcpyDeviceToHost);
    int foundA = (int)count(hitsA.begin(), hitsA.end(), (uint8_t)1);
    cout << "[GPU-A] " << msA << " ms  found " << foundA << "/" << NPATS << "\n";

    vector<PatternRef> byteList[256];
    for (int i = 0; i < NPATS; ++i)
        for (int j = 0; j < (int)needles[i].size(); ++j)
            byteList[(uint8_t)needles[i][j]].push_back({i, j});

    vector<PatternRef> flatIdx;
    vector<int> bounds(257, 0);
    for (int c = 0; c < 256; ++c) {
        bounds[c] = (int)flatIdx.size();
        flatIdx.insert(flatIdx.end(), byteList[c].begin(), byteList[c].end());
    }
    bounds[256] = (int)flatIdx.size();

    vector<int> voteHost(NPATS * HSIZE);
    for (int i = 0; i < NPATS; ++i)
        fill(voteHost.begin() + i * HSIZE,
             voteHost.begin() + i * HSIZE + HSIZE,
             (int)needles[i].size());

    uint8_t    *dHay2;
    PatternRef *dIdx;
    int        *dBounds, *dVote;
    cudaMalloc(&dHay2,   HSIZE * sizeof(uint8_t));
    cudaMalloc(&dIdx,    flatIdx.size() * sizeof(PatternRef));
    cudaMalloc(&dBounds, 257 * sizeof(int));
    cudaMalloc(&dVote,   (size_t)NPATS * HSIZE * sizeof(int));
    cudaMemcpy(dHay2,   hay.data(),        HSIZE * sizeof(uint8_t),                cudaMemcpyHostToDevice);
    cudaMemcpy(dIdx,    flatIdx.data(),    flatIdx.size() * sizeof(PatternRef),    cudaMemcpyHostToDevice);
    cudaMemcpy(dBounds, bounds.data(),     257 * sizeof(int),                      cudaMemcpyHostToDevice);
    cudaMemcpy(dVote,   voteHost.data(),   (size_t)NPATS * HSIZE * sizeof(int),    cudaMemcpyHostToDevice);

    cudaEventRecord(evB0);
    kernelVoteMatrix<<<(HSIZE + TPB - 1) / TPB, TPB>>>(
        dHay2, HSIZE, dIdx, dBounds, dVote, NPATS);
    cudaEventRecord(evB1);
    cudaEventSynchronize(evB1);

    float msB; cudaEventElapsedTime(&msB, evB0, evB1);
    cudaMemcpy(voteHost.data(), dVote, (size_t)NPATS * HSIZE * sizeof(int), cudaMemcpyDeviceToHost);

    vector<uint8_t> hitsB(NPATS, 0);
    for (int i = 0; i < NPATS; ++i) {
        int L = (int)needles[i].size();
        for (int j = 0; j <= HSIZE - L; ++j)
            if (voteHost[i * HSIZE + j] == 0) { hitsB[i] = 1; break; }
    }
    int foundB = (int)count(hitsB.begin(), hitsB.end(), (uint8_t)1);
    cout << "[GPU-B] " << msB << " ms  found " << foundB << "/" << NPATS << "\n\n";

    cout << "=========================================\n  Verification\n=========================================\n";
    cout << "Kernel-A vs CPU : " << (cpuHits == hitsA ? "+ OK" : "- FAIL") << "\n";
    cout << "Kernel-B vs CPU : " << (cpuHits == hitsB ? "+ OK" : "- FAIL") << "\n\n";

    cout << "=========================================\n  Timings\n=========================================\n";
    cout << "CPU    : " << cpuMs << " ms\n";
    cout << "GPU-A  : " << msA   << " ms  (x" << cpuMs/msA   << ")\n";
    cout << "GPU-B  : " << msB   << " ms  (x" << cpuMs/msB   << ")\n";

    cudaFree(dHay);  cudaFree(dPats); cudaFree(dLens); cudaFree(dOutA);
    cudaFree(dHay2); cudaFree(dIdx);  cudaFree(dBounds); cudaFree(dVote);
    cudaEventDestroy(evA0); cudaEventDestroy(evA1);
    cudaEventDestroy(evB0); cudaEventDestroy(evB1);

    return 0;
}



Overwriting search.cu


In [13]:
!nvcc -o search search.cu && ./search

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
  Multi-Pattern Search  |  CUDA Lab

Device : Tesla T4
CC     : 7.5
VRAM   : 14912 MB

Haystack : 100000 B
Patterns : 2000
Lengths  : 3 – 15 B

[CPU]   829.97 ms  found 1600/2000

[GPU-A] 155.78 ms  found 1600/2000
[GPU-B] 10.13 ms  found 1600/2000

  Verification
Kernel-A vs CPU : + OK
Kernel-B vs CPU : + OK

  Timings
CPU    : 829.97 ms
GPU-A  : 155.78 ms  (x5.33)
GPU-B  : 10.13 ms  (x81.90)
